In [1]:
import sys
!{sys.executable} -m pip install -q pydantic-ai-slim openai mcp-server-time "fastmcp-slim[client]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 928.5/928.5 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.1/80.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.0/170.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 738.6/738.6 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.1/74.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.9/70.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/

In [2]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("groq-api-key")

os.environ["GROQ_API_KEY"]   = secret_value_0
os.environ["BASE_URL"]       = "https://api.groq.com/openai/v1"
os.environ["OPENAI_API_KEY"] = secret_value_0

print("Config ready. BASE_URL:", os.environ["BASE_URL"])

Config ready. BASE_URL: https://api.groq.com/openai/v1


In [3]:
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

models = client.models.list()
print("Connected! Available models:")
for m in models.data[:5]:
    print(" •", m.id)

Connected! Available models:
 • llama-3.3-70b-versatile
 • groq/compound-mini
 • allam-2-7b
 • meta-llama/llama-4-scout-17b-16e-instruct
 • canopylabs/orpheus-v1-english


In [4]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

agent_model = OpenAIChatModel("llama-3.3-70b-versatile", provider=provider)

print("Agent model ready!")

Agent model ready!


In [5]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport

time_server = MCPToolset(
    StdioTransport(
        command="python",
        args=["-m", "mcp_server_time", "--local-timezone=America/New_York"],
    )
)

agent = Agent(
    model=agent_model,
    toolsets=[time_server],
    system_prompt=(
        "You MUST use the get_current_time tool to answer any question about the current date or time. "
        "Never say you don't have access to the time. Always call the tool first, then respond."
    )
)

print("Agent with MCP time server ready!")

Agent with MCP time server ready!


In [6]:
async def run_async(prompt: str) -> str:
    async with agent.run_mcp_servers():
        result = await agent.run(prompt)
        return result.output

In [7]:
answer = await run_async("What's the date today?")
print(answer)

The date today is Sunday, May 31, 2026.


In [8]:
import sys
!{sys.executable} -m pip install -q mcp-server-fetch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.26 requires httpx>=0.28.1, but you have httpx 0.27.2 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.
google-genai 1.68.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.


In [9]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport

time_server = MCPToolset(
    StdioTransport(
        command="python",
        args=["-m", "mcp_server_time", "--local-timezone=America/New_York"],
    )
)

fetch_server = MCPToolset(
    StdioTransport(
        command="python",
        args=["-m", "mcp_server_fetch"],
    )
)

agent = Agent(
    model=agent_model,
    toolsets=[time_server, fetch_server],
    system_prompt=(
        "You are a helpful agent with access to two tools: "
        "get_current_time for date/time questions, and fetch for retrieving web content. "
        "Always use the appropriate tool when needed."
    )
)

print("Agent with time + fetch MCP servers ready!")

Agent with time + fetch MCP servers ready!


In [10]:
answer = await run_async("Fetch the content from https://example.com and summarize it.")
print(answer)

Failed to parse JSONRPC message from server
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/mcp/client/stdio/__init__.py", line 155, in stdout_reader
    message = types.JSONRPCMessage.model_validate_json(line)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pydantic/main.py", line 766, in model_validate_json
    return cls.__pydantic_validator__.validate_json(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
pydantic_core._pydantic_core.ValidationError: 1 validation error for JSONRPCMessage
  Invalid JSON: EOF while parsing a value at line 1 column 0 [type=json_invalid, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Failed to parse JSONRPC message from server
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/mcp/client/stdio/__init__.py", line 155, in stdout_reader
    message = types.JSO

The content of https://example.com is a simple webpage that states it is for use in documentation examples without needing permission, and it should be avoided in operations. It also provides a link to learn more about it on the iana.org website.


In [11]:
answer = await run_async(
    "What is today's date and time? Also fetch https://example.com and give me a one line summary."
)
print(answer)

Today's date and time is Sunday, May 31, 2026, 2:51 AM EDT. The one-line summary of https://example.com is: This domain is for use in illustrative examples.
